In [1]:
import json
import unicodedata
from pathlib import Path
import random
from collections import Counter

DATA_DIR = Path("fever-data/raw")
PROCESSED_DATA_DIR = Path("fever-data/processed")
WIKI_DIR = DATA_DIR / "wiki-pages" / "wiki-pages"

TRAIN_FILE = DATA_DIR / "train.jsonl"
DEV_FILE = DATA_DIR / "shared_task_dev.jsonl"
PAPER_DEV_FILE = DATA_DIR / "paper_dev.jsonl"
PAPER_TEST_FILE = DATA_DIR / "paper_test.jsonl"

In [2]:
def normalize_page_id(page_id):
    """Normalize Unicode page IDs to NFC."""
    if page_id is None:
        return None
    return unicodedata.normalize("NFC", page_id)


def load_jsonl(file_path):
    """Load a JSONL file."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def get_required_pages(data):
    """Collect all unique, non-None page IDs referenced by evidence."""
    pages = set()

    for item in data:
        if item["label"] == "NOT ENOUGH INFO":
            continue

        for group in item.get("evidence", []):
            for annotation in group:
                page_id = normalize_page_id(annotation[2])

                if page_id is not None:
                    pages.add(page_id)

    return pages

In [3]:
wiki_lookup = {
    "Page_Name": {
        0: "First sentence.",
        1: "Second sentence."
    }
}

In [4]:
def build_wiki_lookup(wiki_dir, required_pages):
    wiki_lookup = {}

    wiki_files = list(wiki_dir.glob("*.jsonl"))

    print(f"Scanning {len(wiki_files)} Wikipedia files...")

    for file_path in wiki_files:
        with open(file_path, "r", encoding="utf-8") as f:

            for line in f:
                if not line.strip():
                    continue

                try:
                    page = json.loads(line)
                except json.JSONDecodeError:
                    continue

                page_id = normalize_page_id(page.get("id"))

                if page_id not in required_pages:
                    continue

                sentence_lookup = {}

                for row in page.get("lines", "").split("\n"):
                    if not row.strip():
                        continue

                    parts = row.split("\t", 2)

                    if len(parts) < 2:
                        continue

                    try:
                        sentence_id = int(parts[0].strip())
                    except ValueError:
                        continue

                    sentence_text = parts[1].strip()

                    if sentence_text:
                        sentence_lookup[sentence_id] = sentence_text

                wiki_lookup[page_id] = sentence_lookup

    print(f"Required pages: {len(required_pages)}")
    print(f"Extracted pages: {len(wiki_lookup)}")
    print(f"Missing pages: {len(required_pages - set(wiki_lookup))}")

    return wiki_lookup

In [5]:
def diagnose_incomplete_groups(data, wiki_lookup):
    incomplete_groups = []
    total_groups = 0

    for item in data:
        if item["label"] == "NOT ENOUGH INFO":
            continue

        for group_idx, group in enumerate(item.get("evidence", [])):
            total_groups += 1
            missing = []

            for annotation in group:
                page_id = normalize_page_id(annotation[2])
                sentence_id = annotation[3]

                if page_id not in wiki_lookup:
                    missing.append((page_id, sentence_id))
                elif sentence_id not in wiki_lookup[page_id]:
                    missing.append((page_id, sentence_id))

            if missing:
                incomplete_groups.append({
                    "claim_id": item["id"],
                    "group_index": group_idx,
                    "missing": missing
                })

    print(f"Total evidence groups: {total_groups}")
    print(f"Incomplete evidence groups: {len(incomplete_groups)}")

    if incomplete_groups:
        print("\nExamples:")
        for group in incomplete_groups[:5]:
            print(group)

    return incomplete_groups

In [6]:
def preprocess_fever(data, wiki_lookup):
    processed = []
    incomplete_groups = []

    for item in data:
        label = item["label"]

        # NEI claims have no gold evidence.
        if label == "NOT ENOUGH INFO":
            processed.append({
                "id": item["id"],
                "claim": item["claim"],
                "label": "NEI",
                "evidence_group_id": None,
                "evidence": []
            })
            continue

        # Preserve every alternative evidence group separately.
        for group_idx, group in enumerate(item.get("evidence", [])):
            evidence_sentences = []
            missing = []

            for annotation in group:
                page_id = normalize_page_id(annotation[2])
                sentence_id = annotation[3]

                sentence = (
                    wiki_lookup.get(page_id, {}).get(sentence_id)
                )

                if sentence is None:
                    missing.append((page_id, sentence_id))
                else:
                    evidence_sentences.append({
                        "page_id": page_id,
                        "sentence_id": sentence_id,
                        "text": sentence
                    })

            # Never silently train on a partial gold evidence group.
            if missing:
                incomplete_groups.append({
                    "claim_id": item["id"],
                    "group_index": group_idx,
                    "missing": missing
                })
                continue

            processed.append({
                "id": item["id"],
                "claim": item["claim"],
                "label": label,
                "evidence_group_id": group_idx,
                "evidence": evidence_sentences
            })

    print(f"Processed rows: {len(processed)}")
    print(f"Incomplete groups skipped: {len(incomplete_groups)}")

    return processed, incomplete_groups

In [7]:
train_data = load_jsonl(TRAIN_FILE)
dev_data = load_jsonl(DEV_FILE)
paper_dev_data = load_jsonl(PAPER_DEV_FILE)
paper_test_data = load_jsonl(PAPER_TEST_FILE)

# Collect required pages from both datasets.
required_pages = (
    get_required_pages(train_data) |
    get_required_pages(dev_data)    |
    get_required_pages(paper_dev_data) |
    get_required_pages(paper_test_data)
)

# Build the lookup once.
wiki_lookup = build_wiki_lookup(WIKI_DIR, required_pages)

# Diagnose before preprocessing.
train_incomplete = diagnose_incomplete_groups(train_data, wiki_lookup)
dev_incomplete = diagnose_incomplete_groups(dev_data, wiki_lookup)
paper_dev_incomplete = diagnose_incomplete_groups(paper_dev_data, wiki_lookup)
paper_test_incomplete = diagnose_incomplete_groups(paper_test_data, wiki_lookup)

# Construct processed datasets.
train_processed, train_skipped = preprocess_fever(
    train_data, wiki_lookup
)

dev_processed, dev_skipped = preprocess_fever(
    dev_data, wiki_lookup
)

paper_dev_processed, paper_dev_skipped = preprocess_fever(
    paper_dev_data, wiki_lookup
)

paper_test_processed, paper_test_skipped = preprocess_fever(
    paper_test_data, wiki_lookup
)

Scanning 109 Wikipedia files...
Required pages: 14533
Extracted pages: 14533
Missing pages: 0
Total evidence groups: 221476
Incomplete evidence groups: 0
Total evidence groups: 24725
Incomplete evidence groups: 0
Total evidence groups: 12442
Incomplete evidence groups: 0
Total evidence groups: 12283
Incomplete evidence groups: 0
Processed rows: 257115
Incomplete groups skipped: 0
Processed rows: 31391
Incomplete groups skipped: 0
Processed rows: 15775
Incomplete groups skipped: 0
Processed rows: 15616
Incomplete groups skipped: 0


In [8]:
def build_sentence_pool(wiki_dir, pool_size=250_000, seed=42):
    rng = random.Random(seed)
    reservoir = []
    total_sentences = 0

    wiki_files = list(Path(wiki_dir).glob("*.jsonl"))
    print(f"Scanning {len(wiki_files)} Wikipedia files...")

    for file_path in wiki_files:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    page = json.loads(line)
                except json.JSONDecodeError:
                    continue

                page_id = normalize_page_id(page.get("id"))

                for row in page.get("lines", "").split("\n"):
                    if not row.strip():
                        continue

                    parts = row.split("\t", 2)

                    if len(parts) < 2:
                        continue

                    try:
                        sentence_id = int(parts[0].strip())
                    except ValueError:
                        continue

                    sentence_text = parts[1].strip()

                    if not sentence_text:
                        continue

                    sentence = {
                        "page_id": page_id,
                        "sentence_id": sentence_id,
                        "text": sentence_text
                    }

                    total_sentences += 1

                    if len(reservoir) < pool_size:
                        reservoir.append(sentence)
                    else:
                        j = rng.randint(0, total_sentences - 1)

                        if j < pool_size:
                            reservoir[j] = sentence

    print(f"Total corpus sentences scanned: {total_sentences:,}")
    print(f"Sentences retained in sampling pool: {len(reservoir):,}")

    return reservoir


sentence_pool = build_sentence_pool(
    WIKI_DIR,
    pool_size=250_000,
    seed=42
)

Scanning 109 Wikipedia files...
Total corpus sentences scanned: 25,247,890
Sentences retained in sampling pool: 250,000


In [9]:
# Use only training gold evidence to determine the distribution.
positive_train = [
    row for row in train_processed
    if row["label"] in ("SUPPORTS", "REFUTES")
]

group_size_counts = Counter(
    len(row["evidence"]) for row in positive_train
)

print("Gold evidence group size distribution:")
for size, count in sorted(group_size_counts.items()):
    print(f"{size} sentence(s): {count:,}")

Gold evidence group size distribution:
1 sentence(s): 187,670
2 sentence(s): 29,181
3 sentence(s): 3,114
4 sentence(s): 731
5 sentence(s): 303
6 sentence(s): 167
7 sentence(s): 121
8 sentence(s): 52
9 sentence(s): 39
10 sentence(s): 35
11 sentence(s): 11
12 sentence(s): 6
13 sentence(s): 10
14 sentence(s): 9
15 sentence(s): 2
16 sentence(s): 6
17 sentence(s): 2
19 sentence(s): 3
22 sentence(s): 4
23 sentence(s): 3
25 sentence(s): 1
29 sentence(s): 1
30 sentence(s): 2
32 sentence(s): 1
33 sentence(s): 1
41 sentence(s): 1


In [10]:
def build_classifier_dataset(
    processed_data,
    sentence_pool,
    group_size_counts,
    seed=42
):
    rng = random.Random(seed)

    positive_rows = []
    nei_rows = []

    sizes = list(group_size_counts.keys())
    weights = list(group_size_counts.values())

    for row in processed_data:

        if row["label"] in ("SUPPORTS", "REFUTES"):
            positive_rows.append({
                **row,
                "evidence_source": "gold"
            })

        elif row["label"] == "NEI":
            # Draw a realistic evidence-group size.
            evidence_size = rng.choices(
                sizes,
                weights=weights,
                k=1
            )[0]

            # Sample distinct sentences from the corpus pool.
            sampled_sentences = rng.sample(
                sentence_pool,
                evidence_size
            )

            evidence = [
                {
                    "page_id": sentence["page_id"],
                    "sentence_id": sentence["sentence_id"],
                    "text": sentence["text"]
                }
                for sentence in sampled_sentences
            ]

            nei_rows.append({
                "id": row["id"],
                "claim": row["claim"],
                "label": "NEI",
                "evidence_group_id": None,
                "evidence": evidence,
                "evidence_source": "sampled_corpus"
            })

    final_data = positive_rows + nei_rows
    rng.shuffle(final_data)

    return final_data


train_classifier_data = build_classifier_dataset(
    train_processed,
    sentence_pool,
    group_size_counts
)

dev_classifier_data = build_classifier_dataset(
    dev_processed,
    sentence_pool,
    group_size_counts
)

paper_dev_classifier_data = build_classifier_dataset(
    paper_dev_processed,
    sentence_pool,
    group_size_counts
)

paper_test_classifier_data = build_classifier_dataset(
    paper_test_processed,
    sentence_pool,
    group_size_counts
)

In [11]:
def inspect_classifier_dataset(data, name):
    label_counts = Counter(row["label"] for row in data)

    print(f"\n{name}")
    print("-" * 45)
    print("Total rows:", len(data))
    print("Label distribution:", label_counts)

    print("\nEvidence source distribution:")
    print(Counter(row["evidence_source"] for row in data))

    print("\nEvidence sentence count distribution:")
    print(Counter(len(row["evidence"]) for row in data))

    empty_evidence = sum(
        len(row["evidence"]) == 0 for row in data
    )

    print("\nRows with empty evidence:", empty_evidence)

    assert empty_evidence == 0
    assert all(
        row["evidence_source"] == "gold"
        for row in data
        if row["label"] in ("SUPPORTS", "REFUTES")
    )
    assert all(
        row["evidence_source"] == "sampled_corpus"
        for row in data
        if row["label"] == "NEI"
    )


inspect_classifier_dataset(train_classifier_data, "TRAIN")
inspect_classifier_dataset(dev_classifier_data, "DEV")
inspect_classifier_dataset(paper_dev_classifier_data, "PAPER DEV")
inspect_classifier_dataset(paper_test_classifier_data, "PAPER TEST")


TRAIN
---------------------------------------------
Total rows: 257115
Label distribution: Counter({'SUPPORTS': 161249, 'REFUTES': 60227, 'NEI': 35639})

Evidence source distribution:
Counter({'gold': 221476, 'sampled_corpus': 35639})

Evidence sentence count distribution:
Counter({1: 217941, 2: 33804, 3: 3599, 4: 866, 5: 353, 6: 190, 7: 141, 8: 62, 9: 44, 10: 41, 11: 15, 13: 10, 14: 10, 12: 9, 16: 6, 22: 4, 19: 3, 30: 3, 23: 3, 15: 2, 17: 2, 25: 2, 33: 2, 41: 1, 32: 1, 29: 1})

Rows with empty evidence: 0

DEV
---------------------------------------------
Total rows: 31391
Label distribution: Counter({'SUPPORTS': 12498, 'REFUTES': 12227, 'NEI': 6666})

Evidence source distribution:
Counter({'gold': 24725, 'sampled_corpus': 6666})

Evidence sentence count distribution:
Counter({1: 27287, 2: 3536, 3: 364, 4: 96, 5: 43, 6: 20, 7: 17, 8: 12, 10: 4, 12: 3, 9: 3, 11: 2, 15: 1, 17: 1, 20: 1, 25: 1})

Rows with empty evidence: 0

PAPER DEV
---------------------------------------------
Total 

In [12]:
def save_jsonl(data, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

save_jsonl(
    train_classifier_data,
    PROCESSED_DATA_DIR / "train_classifier.jsonl"
)

save_jsonl(
    dev_classifier_data,
    PROCESSED_DATA_DIR / "dev_classifier.jsonl"
)

save_jsonl(
    paper_dev_classifier_data,
    PROCESSED_DATA_DIR / "paper_dev_classifier.jsonl"
)

save_jsonl(
    paper_test_classifier_data,
    PROCESSED_DATA_DIR / "paper_test_classifier.jsonl"
)

In [13]:
import random
import json

rng = random.Random(42)

nei_samples = rng.sample(
    [x for x in train_classifier_data if x["label"] == "NEI"],
    5
)

for i, row in enumerate(nei_samples, 1):
    print(f"\n{'='*80}")
    print(f"NEI SAMPLE {i}")
    print("Claim:", row["claim"])

    for j, evidence in enumerate(row["evidence"], 1):
        print(f"Evidence {j}: {evidence['text']}")


NEI SAMPLE 1
Claim: James Bond is not a series of spy films.
Evidence 1: Chris Townsend -LRB- writer -RRB- -LRB- born 1949 -RRB- , hillwalker and author
Evidence 2: It controlled the main road that went from the Persian province of Syria to Egypt .

NEI SAMPLE 2
Claim: The Wayans Bros. only aired on CBS Television Network.
Evidence 1: The school was located on Crystal Street in Petersham North , the current site of the Petersham College of the Sydney Institute of TAFE .
Evidence 2: Formerly known as Haines Store , Hainesville once had its own post office in operation .

NEI SAMPLE 3
Claim: Sienna Miller is a student.
Evidence 1: where they raise livestock including cattle and sheep .

NEI SAMPLE 4
Claim: The Coen brothers often undermine or imitate with humorous intent various categories and tropes.
Evidence 1: In 1890 he won the Murchison Scholarship of the Royal College of Physicians .

NEI SAMPLE 5
Claim: James Blake (musician) released The Matrix in 2016.
Evidence 1: Leonidas Papa

In [14]:
for label in ["SUPPORTS", "REFUTES"]:
    samples = rng.sample(
        [x for x in train_classifier_data if x["label"] == label],
        5
    )

    for row in samples:
        print("\nLabel:", label)
        print("Claim:", row["claim"])
        for e in row["evidence"]:
            print("-", e["text"])


Label: SUPPORTS
Claim: Earl Warren held the office of Chief Justice of the United States.
- Earl Warren -LRB- March 19 , 1891 -- July 9 , 1974 -RRB- was an American jurist and politician , who served as the 30th Governor of California -LRB- 1943 -- 1953 -RRB- and later the 14th Chief Justice of the United States -LRB- 1953 -- 1969 -RRB- .

Label: SUPPORTS
Claim: Uzbekistan is a member of the SCO.
- Uzbekistan is a member of the Commonwealth of Independent States -LRB- CIS -RRB- , Organization for Security and Co-operation in Europe -LRB- OSCE -RRB- , UN , and the SCO .

Label: SUPPORTS
Claim: Martin Freeman is known for his portrayal of Lester Nygaard.
- Martin John Christopher Freeman -LRB- born 8 September 1971 -RRB- is an English actor , who became known for portraying Tim Canterbury in the original UK version of sitcom mockumentary The Office , Dr. John Watson in the British crime drama Sherlock , Bilbo Baggins in Peter Jackson 's The Hobbit film trilogy , and Lester Nygaard in th